### Final Project CSE 476
#### Group members: Neil Shenoy, Kavish Sharma, Jenna Skabelund

In [ ]:
%pip install requests python-dotenv

import os, json, textwrap, re, time
import requests

API_KEY = "sk-THSHSCeQUCzILV98j-3xCQ"
API_BASE = os.getenv("API_BASE", "https://openai.rc.asu.edu/v1")  
MODEL    = os.getenv("MODEL_NAME", "qwen3-30b-a3b-instruct-2507")              

def call_model_chat_completions(prompt: str,
                                system: str = "You are a helpful assistant. Reply with only the final answer—no explanation.",
                                model: str = MODEL,
                                temperature: float = 0.0,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 128,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}


In [ ]:
#Helper

def get_response_text(response_dict):

    if not isinstance(response_dict, dict):
        return ""
    return (response_dict.get("text") or "").strip()

In [ ]:
#ask for an answer, then ask the model "is this answer correct? if not, fix it." Two API calls total.

def solve_with_reflection(question, model=None, temperature=0.0, too_wordy=False):



    first_system = (
        "You are great at finding the correct answers to any given questions. "
        "Solve the user's question and reply with the best final answer you can."
    )

    first_prompt = f"""Question: {question} What is the answer?"""
    first_response = call_model_chat_completions(
        first_prompt,
        system=first_system,
        model=model,
        temperature=temperature,
    )
    initial_answer = get_response_text(first_response)



    reflection_system = (
        "You are great at reviewing questions. "
        "Check whether the proposed answer is correct and complete. "
        "If it is wrong, fix it. "
        "Return only the correct and complete final answer."
    )

    reflection_prompt = f"""You are reviewing a previous answer. Original question: {question} Previous answer: {initial_answer} Task: Check the answer carefully. If it is correct, keep it. If it is wrong or incomplete, fix it. Return the corrected final answer only."""

    reflection_response = call_model_chat_completions(
        reflection_prompt,
        system=reflection_system,
        model=model,
        temperature=0.0,
    )
    reflected_answer = get_response_text(reflection_response)
    final_answer = reflected_answer if reflected_answer else initial_answer

    if too_wordy:
        print("QUESTION:", question)
        print("INITIAL ANSWER:", initial_answer)
        print("NEW REFLECTED ANSWER:", reflected_answer)
        print("FINAL ANSWER:", final_answer)

    return {
        "initial_answer": initial_answer,
        "reflected_answer": reflected_answer,
        "final_answer": final_answer,
    }

In [ ]:
#Test

example_question = "What is 17 * 26?"
reflection_result = solve_with_reflection(example_question, too_wordy=True)